## Before running

In order to keep the GA code consistently usable on Avon, I have created a virtual environment consistent with the python packages on Avon. To use this environment you need to install pipenv with 
- 'pip install pipenv'

Then to create a virtual environment with all the required packages run
- 'pipenv install'

To activate the environment run
- 'pipenv shell'

You can now run the GA code. To exit the virtual environment just use
- 'exit'

All the required packages are listed in 'SOAP_GAS_TMS/Pipfile'

In [ ]:
from genetic_algorithm import *

## Inputs

Dictionaries are taken as input from a parameter file, they contain the parameters for each soap descriptor

In [ ]:
descDict1 = {'lower': 1, 'upper': 10, 'centres': '{6}',
             'neighbours': '{6}', 'mu': 0, 
             'mu_hat': 0, 'nu': 2, 'nu_hat': 0, 'mutation_chance': 0.50, 
             'min_cutoff': 4, 'max_cutoff': 10, 'min_sigma': 0.1, 
             'max_sigma': 0.9, 'message_steps': 0}

descDict2 = {'lower': 1, 'upper': 5, 'centres': '{6}',
             'neighbours': '{6}', 'mu': 0, 
             'mu_hat': 0, 'nu': 2, 'nu_hat': 0, 'mutation_chance': 0.50, 
             'min_cutoff': 4, 'max_cutoff': 5, 'min_sigma': 0.1, 
             'max_sigma': 0.9, 'message_steps': 0}

Other parameters are also taken as input. These are automatically checked that the parameters are viable

In [ ]:
num_gens = 5
best_sample, lucky_few, population_size, number_of_children = 4, 2, 12, 4
early_stop = 2
early_number = 3 
min_generations = 5

## GeneParameter

GeneParameter class is created from each descriptor dictionary. 

In [ ]:
params1 = GeneParameters(**descDict1)
params2 = GeneParameters(**descDict2)

In [ ]:
params1

## GeneSet

We can use these classes to create a specific set of parameters that are consistant with these values. This returns a randomly generated GeneSet class

In [ ]:
example_gene_set = params1.make_gene_set()
example_gene_set

We can get the parameters used to create the GeneSet class

In [ ]:
example_gene_set.gene_parameters

We can get a descriptor string to be used as an input for getting SOAPs

In [ ]:
example_gene_set.get_soap_string()

We can also mutate the gene using the mutation chance in the GeneParameters class

In [ ]:
print(f"Before mutation {example_gene_set}")
example_gene_set.mutate_gene()
print(f"After mutation {example_gene_set}")

## Individual

An Individual is made up of a list of GeneSet classes.

In [ ]:
example_gene_set_two = params2.make_gene_set()
gene_set_list = [example_gene_set, example_gene_set_two]
example_individual = Individual(gene_set_list[:1])
example_individual

Getting the score for an indivudual

In [ ]:
example_individual.get_score()
example_individual.score

Breeding two individuals to create a child. Mutation is automatically performed during this

In [ ]:
example_individual_two = Individual(gene_set_list)
print(f"Breeding {example_individual} with {example_individual_two}")
child = breed_individuals(example_individual, example_individual_two)
print(f"Created child {child}")

## Population

A Population is a collection of Individual classes. This can be created using a list of GeneParameter classes

In [ ]:
gene_parameters = [params1, params2]
pop = Population(best_sample, lucky_few, population_size, 
                 number_of_children, gene_parameters, 
                 maximise_scores = True)
pop

To initialise the population

In [ ]:
pop.initialise_population()

If you want a way of neatly seeing what individuals are in the population

In [ ]:
pop.print_population()

The next generation can then be generated 

In [ ]:
pop.next_generation()
pop.print_population()

So to run the full GA 

In [ ]:
for _ in range(num_gens):
    pop.next_generation()
pop.print_population()

## BestHistory

BestHistory is a class to store the history and check convergence criteria. So the entire GA can be run, printed, and saved using the following code snippet:

In [ ]:
hist = BestHistory(early_stop, early_number, min_generations)
pop = Population(best_sample, lucky_few, population_size, 
                 number_of_children, gene_parameters, 
                 maximise_scores = True)

for gen in range(num_gens):
    if hist.converged:
        break
    print(f"Generation {gen}")
    pop.next_generation()
    hist.append(pop)
    write_to_outfile("-------")

There now exists the entire history of the best Individuals throughout each generation that can be saved and easily accessed. 

In [ ]:
vars(hist)

### Running the GA as a script

To run the GA as a script, you just need an input file. See 'SOAP_GAS_TMS/input.py' for an example of the format. 

Then to run the GA with this input file, run 'python genetic_algorithm.py \<input file name without .py extension>'

Running the script creates two files, out_\<input file name>.txt and history_\<input file name>.pkl

If these files already exist, the old ones will be backed up and new ones created. 

The out file contains a log of everything happening in the GA. 
The history file contains a BestHistory object containing the best individuals throughout the GA.